In [22]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [23]:
# !pip install -q langchain-openai langchain-core langchain-community pydantic

import os
from typing import Optional
from pydantic import BaseModel, ValidationError
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.runnables import RunnableLambda, RunnableMap

In [24]:
# Set your key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [25]:
class OrderQuery(BaseModel):
    """
    Structured representation of a customer's query.
    All fields are optional to allow flexible parsing,
    but we will apply logic later to enforce what is required.
    """
    order_id: Optional[int] = None
    product_name: Optional[str] = None
    issue_type: Optional[str] = None  # e.g. "return", "status", "recommendation"
    details: Optional[str] = None


In [26]:
response_schemas = [
    ResponseSchema(
        name="order_id",
        description="The numeric order ID if mentioned by the user, otherwise null."
    ),
    ResponseSchema(
        name="product_name",
        description="Product name or category mentioned by the user, otherwise null."
    ),
    ResponseSchema(
        name="issue_type",
        description='High-level type of issue: "status", "return", "recommendation", "other".'
    ),
    ResponseSchema(
        name="details",
        description="Any additional free-text details about the issue or user request."
    ),
]

order_query_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = order_query_parser.get_format_instructions()


In [27]:
order_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI assistant for an e-commerce customer support chatbot. "
            "You MUST extract structured information about the customer's request.\n\n"
            "{format_instructions}"
        ),
        (
            "human",
            "Customer message: {user_query}"
        ),
    ]
)


In [28]:
# Chain: prompt -> LLM -> StructuredOutputParser
extract_order_query_chain = (
    order_query_prompt
    | llm
    | order_query_parser
)


In [29]:
test_queries = [
    "Where is my order #12345?",
    "Suggest me a phone under ₹20,000.",
    "I want to return my shoes from order 98765; they are damaged.",
]

for q in test_queries:
    print("USER:", q)
    parsed = extract_order_query_chain.invoke(
        {"user_query": q, "format_instructions": format_instructions}
    )
    print("PARSED:", parsed, "\n")


USER: Where is my order #12345?
PARSED: {'order_id': '12345', 'product_name': None, 'issue_type': 'status', 'details': 'Customer is inquiring about the status of their order.'} 

USER: Suggest me a phone under ₹20,000.
PARSED: {'order_id': None, 'product_name': 'phone', 'issue_type': 'recommendation', 'details': 'Suggest me a phone under ₹20,000.'} 

USER: I want to return my shoes from order 98765; they are damaged.
PARSED: {'order_id': '98765', 'product_name': 'shoes', 'issue_type': 'return', 'details': 'The shoes are damaged.'} 



In [30]:
def validate_order_query(parsed_dict: dict) -> OrderQuery:
    """
    Validate the parsed LLM output using the OrderQuery Pydantic model.
    Raises ValidationError if data is invalid.
    """
    return OrderQuery(**parsed_dict)

In [31]:
def _validate_step(parsed: dict):
    try:
        obj = validate_order_query(parsed)
        return {"success": True, "order_query": obj, "error": None}
    except ValidationError as e:
        # You can log e.errors() in a real system
        return {
            "success": False,
            "order_query": None,
            "error": f"Validation failed: {e}"
        }

validate_step = RunnableLambda(_validate_step)

order_understanding_chain = (
    order_query_prompt
    | llm
    | order_query_parser
    | validate_step
)


In [32]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [33]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful retail customer support agent for QuickShop.\n"
            "You extract structured information from the conversation and "
            "keep track of the order_id across turns.\n\n"
            "{format_instructions}"
        ),
        ("system", "Conversation so far:\n{chat_history}"),
        ("human", "{user_query}"),
    ]
)

chat_chain_core = (
    chat_prompt
    | llm
    | order_query_parser
    | validate_step
)


In [34]:
# Simple in-memory store: session_id -> ChatMessageHistory
store = {}

def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat_chain_with_memory = RunnableWithMessageHistory(
    chat_chain_core,
    get_history,
    input_messages_key="user_query",
    history_messages_key="chat_history",
)


In [35]:
def chat(session_id: str, user_query: str):
    result = chat_chain_with_memory.invoke(
        {
            "user_query": user_query,
            "format_instructions": format_instructions,
        },
        config={"configurable": {"session_id": session_id}},
    )
    return result


In [36]:
session_id = "demo-user-1"

print("TURN 1: Ask for order status")
res1 = chat(session_id, "Where is my order #12345?")
print(res1, "\n")

print("TURN 2: Start a return without mentioning the order id again")
res2 = chat(session_id, "I want to return it. The product arrived damaged.")
print(res2, "\n")


TURN 1: Ask for order status


Error in RootListenersTracer.on_chain_end callback: KeyError('output')


{'success': True, 'order_query': OrderQuery(order_id=12345, product_name=None, issue_type='status', details='User is inquiring about the status of their order.'), 'error': None} 

TURN 2: Start a return without mentioning the order id again


Error in RootListenersTracer.on_chain_end callback: KeyError('output')


{'success': True, 'order_query': OrderQuery(order_id=None, product_name=None, issue_type='return', details='The product arrived damaged.'), 'error': None} 

